# 02 - Governed SQL Agent With LangGraph

A minimal governed-SQL pattern: every draft query is validated with Metatate,
and the `verdict` routes the agent — `pass` approves, `warn` revises to a
minimized query, `fail` blocks. The same routing runs as a real LangGraph
`StateGraph` in `framework_runtime/langgraph_acceptance.py`.


In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from common import get_client

mode = os.getenv("METATATE_EXAMPLES_MODE", "offline")
if mode == "live" and not os.getenv("METATATE_MCP_URL"):
    print("Live mode needs a Metatate endpoint. Fastest path (about 5 minutes):")
    print("  1. Create a free account: https://app.getmetatate.com/sign-up?ref=examples")
    print("  2. Workspace dashboard: 'Load the demo' banner -> 'Load the AcmeCloud demo'")
    print("  3. MCP Tools -> Tokens: issue a token; Connect tab has your endpoint URL")
    print("  4. export METATATE_MCP_URL=... METATATE_SAAS_MCP_TOKEN=...")
    print("     (full steps: docs/live-mode-saas.md)")

client = get_client()
print(f"Metatate examples mode: {mode}")


def asset(table, column=None):
    ref = {"database": "acmecloud_demo", "schema": "public", "table": table}
    if column:
        ref["column"] = column
    return ref


def answer_label(answer):
    state = answer.get("state")
    if state and state != "answered":
        return state
    return answer.get("decision") or answer.get("verdict") or "unknown"


def print_answer(answer):
    print(f"state:    {answer.get('state')}")
    if "decision" in answer:
        print(f"decision: {answer['decision']}")
    if "verdict" in answer:
        print(f"verdict:  {answer['verdict']}")
    if answer.get("reason"):
        print(f"reason:   {answer['reason']}")
    for condition in answer.get("conditions") or []:
        print(f"condition [{condition.get('kind')}]: {condition.get('requirement')}")
    for prohibition in answer.get("prohibitions") or []:
        print(f"prohibition: {prohibition.get('detail')}")
    for obligation in answer.get("obligations") or []:
        print(f"obligation [{obligation.get('type')}]: {obligation.get('target')}")
    if "can_proceed_now" in answer:
        print(f"can_proceed_now: {answer['can_proceed_now']}")


In [ ]:
SAFE_SQL = "SELECT region, SUM(arr) FROM customers GROUP BY region"

def governed_sql(sql, scenario_key):
    answer = client.validate_query_context(
        sql,
        scenario_key=scenario_key,
        default_database="acmecloud_demo",
        default_schema="public",
    )
    verdict = answer["verdict"]
    if verdict == "fail":
        return {"verdict": verdict, "final_sql": None, "route": "block"}
    if verdict == "warn":
        return {"verdict": verdict, "final_sql": SAFE_SQL, "route": "revise"}
    return {"verdict": verdict, "final_sql": sql, "route": "approve"}


In [ ]:
runs = {
    "safe": governed_sql(SAFE_SQL, "purpose.allowed_use"),
    "unsafe": governed_sql(
        "SELECT customer_name, email FROM customers WHERE region = 'EU'",
        "purpose.allowed_use",
    ),
    "blocked": governed_sql(
        "SELECT customer_name, email FROM customers WHERE marketing_consent = 'opted_in'",
        "purpose.prohibited_use",
    ),
}
for name, run in runs.items():
    print(f"{name}: {run['route']} ({run['verdict']}) -> {run['final_sql']}")


The deterministic runtime proof (a real `StateGraph` with approve/revise/block
routing) lives in `framework_runtime/` and runs in CI — see
`docs/framework-runtime-acceptance.md`.
